<a href="https://colab.research.google.com/github/smkim0508/cos484-notes/blob/main/A2P1_Named_Entity_Recognition_HMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programming Problem 1: HMM for NER
Welcome to the programming portion of the assignment! Each assignment throughout the semester will have a written portion and a programming portion. We will be using [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb#recent=true), so if you have never used it before, take a quick look through this introduction: [Working with Google Colab](https://docs.google.com/document/d/1LlnXoOblXwW3YX-0yG_5seTXJsb3kRdMMRYqs8Qqum4/edit?usp=sharing).

### Writing Code
Look for the keyword "TODO" and fill in your code in the empty space.
Feel free to add and delete arguments in function signatures, but be careful that you might need to change them in function calls which are already present in the notebook.

### Data preprocessing

In this section we will write code to load data and build the dataset for Named Entity Recognition.

You may inspect the data first before writing the data preprocessing code by looking at the data file: https://princeton-nlp.github.io/cos484/assignments/a2/eng.train. Hints on processing the data:
- You may ignore the lines with "DOCSTART"
- Examples of NER tags include "O", "ORG", "MISC", and it's always in the same position in each line of the data.
- To process numbers more easily, you can replace all digits with 0's (to avoid out-of-vocab words)

You should end up with a list of sentences, where each sentence is represented with a list of words and tags.

Additional, you will want to support the following functions for later:
- Map words and tags to ids (integers)
- Handle unknown words in mapping

In [ ]:
import re
from typing import Optional

class NERDataLoader():
    """
    Loads the NER dataset into a list of sentences.
    Each sentence is in the form: (words: list[str], tags: list[str])
    - We handle a list of the above tuples

    NOTE:
    - Skips lines containing "DOCSTART"
    - Tag is taken as the last whitespace-separated field on each line
    - Replaces digits with '0' to reduce OOV cases, as guided in problem statement
    - Builds word/tag -> id maps from the training split
    - Unknown words map to <UNK>
    """

    PAD = "<PAD>"
    UNK = "<UNK>"

    # main class init
    def __init__(self, normalize_digits: bool = True):
        self.normalize_digits = normalize_digits
        self._digit_re = re.compile(r"\d")

        self.word_to_id: dict[str, int] = {self.PAD: 0, self.UNK: 1}
        self.tag_to_id: dict[str, int] = {self.PAD: 0}

        self.train: list[tuple[list[str], list[str]]] = []
        self.valid: list[tuple[list[str], list[str]]] = []

    # helper to load in data and parse it
    # NOTE: uses is_train to indicate whether data is train data or val data
    def load(self, path: str, *, is_train: bool) -> None:
        sentences = self._read_file(path)
        if is_train:
            self.train = sentences
        else:
            self.valid = sentences

    # builds internal map of words & tags to ids
    def build_id_maps(self) -> None:
        if not self.train:
            print(f"Training data doesn't exist. Please run load() with is_train == True")
            return

        for words, tags in self.train:
            for w in words:
                if w not in self.word_to_id:
                    self.word_to_id[w] = len(self.word_to_id)
            for t in tags:
                if t not in self.tag_to_id:
                    self.tag_to_id[t] = len(self.tag_to_id)

    # encodes input words using the map
    def encode(self, words: list[str], tags: Optional[list[str]] = None):
        w_ids = [self.word_to_id.get(w, self.word_to_id[self.UNK]) for w in words]
        if tags is None:
            return w_ids
        t_ids = [self.tag_to_id[t] for t in tags]
        return w_ids, t_ids

    # helper to parse file and update internal states
    def _read_file(self, path: str) -> list[tuple[list[str], list[str]]]:
        sentences: list[tuple[list[str], list[str]]] = []
        cur_words: list[str] = []
        cur_tags: list[str] = []

        # flushes existing words
        def flush():
            nonlocal cur_words, cur_tags
            if cur_words:
                sentences.append((cur_words, cur_tags))
                cur_words, cur_tags = [], []

        with open(path, "r", encoding="utf-8") as f:
            for raw in f:
                line = raw.strip()

                if not line:
                    flush()
                    continue

                if "DOCSTART" in line:
                    continue

                parts = line.split()
                word = parts[0]
                tag = parts[-1]

                if self.normalize_digits:
                    word = self._digit_re.sub("0", word)

                cur_words.append(word)
                cur_tags.append(tag)

        flush()
        return sentences

In [ ]:
# download the provided data for a quick test

import urllib.request

url = "https://princeton-nlp.github.io/cos484/assignments/a2/eng.train"
out_path = "eng.train"

urllib.request.urlretrieve(url, out_path)
print("saved to", out_path)

saved to eng.train


In [ ]:
# verify that the loader is working properly

loader = NERDataLoader(normalize_digits=True)
loader.load("eng.train", is_train=True)
loader.build_id_maps()

print("# sentences:", len(loader.train))
print("# word types:", len(loader.word_to_id))
print("# tag types:", len(loader.tag_to_id))
print("tags:", sorted([t for t in loader.tag_to_id.keys() if t != loader.PAD]))

# sentences: 14041
# word types: 20102
# tag types: 6
tags: ['LOC', 'MISC', 'O', 'ORG', 'PER']


### Hidden Markov Model
In this section, we will implement a bigram hidden markov model (HMM) that could perform two types of decoding: greedy decoding and viterbi decoding.

Specifically, you should include the following functionalities:
- Initialize the HMM given the word and tag mappings.
- Train the HMM with a given corpus
- Greedy decoding: given a single sentence, output its tags with greedy algorithm
- Viterbi decoding: given a single sentence, output its tags using Viterbi

You may refer to the lecture notes for more details on the HMM and the decoding algorithms.

In [ ]:
import math
from typing import Optional

# define -inf to use later
NEG_INF = -float('inf')

class BigramHMM():
    """
    Bigram HMM for NER, learning from:
      - transition probs: p(tag_t | tag_{t-1})
      - emission probs: p(word_t | tag_t)

    Decodes with greedy and viterbi approaches.

    NOTE:
      - Uses <START> and <STOP> internally as tags (not emitted).
      - Uses add-alpha smoothing.
    """

    # replaces <s> and </s> tokens
    START = "<START>"
    STOP = "<STOP>"

    def __init__(
        self,
        word_to_id: dict[str, int],
        tag_to_id: dict[str, int],
        *,
        unk_token: str = "<UNK>", # NOTE: unknown tokens used to handle edge case during testing
        # hyperparams
        alpha_trans: float = 1e-3,
        alpha_emit: float = 1e-3,
    ):
        self.word_to_id = dict(word_to_id)
        self.tag_to_id = dict(tag_to_id)
        self.unk_token = unk_token
        self.alpha_trans = alpha_trans
        self.alpha_emit = alpha_emit

        # ensure START/STOP exists
        if self.START not in self.tag_to_id:
            self.tag_to_id[self.START] = len(self.tag_to_id)
        if self.STOP not in self.tag_to_id:
            self.tag_to_id[self.STOP] = len(self.tag_to_id)

        self.id_to_tag = {i: t for t, i in self.tag_to_id.items()}

        self.start_id = self.tag_to_id[self.START]
        self.stop_id = self.tag_to_id[self.STOP]

        self.T = len(self.tag_to_id)
        self.V = len(self.word_to_id)

        self.logA: Optional[list[list[float]]] = None
        self.logB: Optional[list[list[float]]] = None

    # training method
    def fit(self, train: list[tuple[list[str], list[str]]]) -> None:
        # counts
        trans = [[0] * self.T for _ in range(self.T)]
        trans_den = [0] * self.T

        emit = [[0] * self.V for _ in range(self.T)]
        emit_den = [0] * self.T

        def wid(w: str) -> int:
            return self.word_to_id.get(w, self.word_to_id[self.unk_token])

        def tid(t: str) -> int:
            return self.tag_to_id[t]

        for words, tags in train:
            prev = self.start_id
            for w, t in zip(words, tags):
                y = tid(t)
                x = wid(w)

                trans[prev][y] += 1
                trans_den[prev] += 1

                emit[y][x] += 1
                emit_den[y] += 1

                prev = y

            # STOP transition
            trans[prev][self.stop_id] += 1
            trans_den[prev] += 1

        # log transitions
        aT = self.alpha_trans
        self.logA = [[0.0] * self.T for _ in range(self.T)]
        for prev in range(self.T):
            denom = trans_den[prev] + aT * self.T
            for y in range(self.T):
                p = (trans[prev][y] + aT) / denom
                self.logA[prev][y] = math.log(p)

        # log emissions
        aE = self.alpha_emit
        self.logB = [[0.0] * self.V for _ in range(self.T)]
        for y in range(self.T):
            denom = emit_den[y] + aE * self.V
            if denom == 0:
                # START/STOP (or unseen tag) don't emit; define uniform for safety
                uni = -math.log(self.V)
                for x in range(self.V):
                    self.logB[y][x] = uni
            else:
                for x in range(self.V):
                    p = (emit[y][x] + aE) / denom
                    self.logB[y][x] = math.log(p)

    # decoding helpers below:
    def greedy_decode(self, words: list[str]) -> list[int]:
        assert self.logA is not None and self.logB is not None

        def wid(w: str) -> int:
            return self.word_to_id.get(w, self.word_to_id[self.unk_token])

        path: list[int] = []
        prev = self.start_id

        for w in words:
            x = wid(w)
            best_y = -1
            best_score = NEG_INF

            for y in range(self.T):
                if y == self.start_id or y == self.stop_id:
                    continue
                score = self.logA[prev][y] + self.logB[y][x]
                if score > best_score:
                    best_score = score
                    best_y = y

            path.append(best_y)
            prev = best_y

        return path

    def viterbi_decode(self, words: list[str]) -> list[int]:
        assert self.logA is not None and self.logB is not None
        n = len(words)
        if n == 0:
            return []

        def wid(w: str) -> int:
            return self.word_to_id.get(w, self.word_to_id[self.unk_token])

        dp = [[NEG_INF] * self.T for _ in range(n)]
        bp = [[-1] * self.T for _ in range(n)]

        # init from START
        x0 = wid(words[0])
        for y in range(self.T):
            if y == self.start_id or y == self.stop_id:
                continue
            dp[0][y] = self.logA[self.start_id][y] + self.logB[y][x0]
            bp[0][y] = self.start_id

        # DP
        for t in range(1, n):
            x = wid(words[t])
            for y in range(self.T):
                if y == self.start_id or y == self.stop_id:
                    continue

                best_prev = -1
                best_val = NEG_INF
                for prev in range(self.T):
                    val = dp[t - 1][prev] + self.logA[prev][y] + self.logB[y][x]
                    if val > best_val:
                        best_val = val
                        best_prev = prev

                dp[t][y] = best_val
                bp[t][y] = best_prev

        # end with STOP
        best_last = -1
        best_score = NEG_INF
        for y in range(self.T):
            if y == self.start_id or y == self.stop_id:
                continue
            score = dp[n - 1][y] + self.logA[y][self.stop_id]
            if score > best_score:
                best_score = score
                best_last = y

        # backtrack
        path = [best_last]
        for t in range(n - 1, 0, -1):
            path.append(bp[t][path[-1]])
        path.reverse()
        return path

    def decode_ids(self, tag_ids: list[int]) -> list[str]:
        return [self.id_to_tag[i] for i in tag_ids]

### Train and evaluate HMMs.

In this section, you will implement the logic for training and evaluating the HMMs:
- Train the model by calling the functions/classes you implemented above,
- Evaluate the trained model on the training and evaluation set by calculating the accuracy of the predicted tags.
- Compute the confusion matrix and F1 score of the predictions.

In [ ]:
class HMMTrainer():
    """
    Minimal train/eval helpers and utils for BigramHMM on the NER dataset.
      - token accuracy
      - per-class precision / recall / F1 (for the 5 tags, including "O")
      - confusion matrix (rows=true, cols=pred)
    """

    def __init__(
        self,
        loader: NERDataLoader,
        *,
        drop_pad_tag: bool = True,
        alpha_trans: float = 1e-3,
        alpha_emit: float = 1e-3,
    ):
        self.loader = loader
        self.drop_pad_tag = drop_pad_tag
        self.alpha_trans = alpha_trans
        self.alpha_emit = alpha_emit

        self.model: BigramHMM | None = None

    # main training helper that produces hmm fit on data
    def train(self) -> BigramHMM:
        # NOTE: using raise to prevent missing data cases
        if not self.loader.train:
            raise ValueError("No training data found. Call loader.load(..., split='train') first.")
        if len(self.loader.word_to_id) <= 2 or len(self.loader.tag_to_id) <= 1:
            raise ValueError("ID maps not built. Call loader.build_id_maps() first.")

        tag_to_id = self._make_tag_map()

        hmm = BigramHMM(
            word_to_id=self.loader.word_to_id,
            tag_to_id=tag_to_id,
            unk_token=self.loader.UNK,
            alpha_trans=self.alpha_trans,
            alpha_emit=self.alpha_emit,
        )
        hmm.fit(self.loader.train)
        self.model = hmm
        return hmm

    # eval helper, takes decoding method as arg
    def evaluate(self, split: str, *, decode: str = "greedy") -> dict:
        if self.model is None:
            raise ValueError("Model not trained. Call train() first.")

        data = self._get_split(split)
        decode_fn = self.model.greedy_decode if decode == "greedy" else self.model.viterbi_decode
        label_names, label_ids, id_to_col = self._labels()

        cm = [[0] * len(label_ids) for _ in range(len(label_ids))]
        correct = 0
        total = 0

        for words, gold_tags in data:
            if not words:
                continue

            pred_ids = decode_fn(words)

            # safety: if something goes wrong, keep lengths aligned
            n = min(len(gold_tags), len(pred_ids))
            for i in range(n):
                g = self.model.tag_to_id[gold_tags[i]]
                p = pred_ids[i]

                # if decoder somehow ever outputs a non-label state (it shouldn't), map it to "O" if possible
                if p not in id_to_col:
                    p = self._fallback_label_id()

                gi = id_to_col[g]
                pi = id_to_col[p]
                cm[gi][pi] += 1

                total += 1
                if g == p:
                    correct += 1

        acc = (correct / total) if total else 0.0
        per_label = self._f1_from_confusion(cm, label_names)

        return {
            "split": split,
            "decode": decode,
            "accuracy": acc,
            "labels": label_names,
            "confusion_matrix": cm, # NOTE: rows=truth, cols=pred
            "per_label": per_label, # {label: {precision, recall, f1, support}}
        }

    # utils below:

    def _get_split(self, split: str) -> list[tuple[list[str], list[str]]]:
        if split == "train":
            return self.loader.train
        if split in ("valid", "val", "dev"):
            return self.loader.valid
        raise ValueError(f"split must be 'train' or 'valid/val/dev', got {split!r}")

    def _make_tag_map(self) -> dict[str, int]:
        # Optionally drop <PAD> so the HMM never predicts it.
        if not self.drop_pad_tag:
            return dict(self.loader.tag_to_id)

        pad = self.loader.PAD
        items = [(t, i) for t, i in self.loader.tag_to_id.items() if t != pad]
        items.sort(key=lambda x: x[1])  # keep stable order
        return {t: j for j, (t, _) in enumerate(items)}

    def _labels(self):
        assert self.model is not None
        specials = {self.model.START, self.model.STOP}

        # label order by tag id (stable)
        items = [(t, i) for t, i in self.model.tag_to_id.items() if t not in specials]
        items.sort(key=lambda x: x[1])

        label_names = [t for t, _ in items]
        label_ids = [i for _, i in items]
        id_to_col = {tid: j for j, tid in enumerate(label_ids)}
        return label_names, label_ids, id_to_col

    def _fallback_label_id(self) -> int:
        # map unexpected predictions to "O" when possible
        assert self.model is not None
        if "O" in self.model.tag_to_id:
            return self.model.tag_to_id["O"]
        # otherwise just pick the smallest non-special id
        label_names, label_ids, _ = self._labels()
        return label_ids[0]

    # finds f1 and other evaluation metrics from the cm
    def _f1_from_confusion(self, cm: list[list[int]], labels: list[str]) -> dict[str, dict[str, float]]:
        L = len(labels)
        row_sums = [sum(cm[i]) for i in range(L)]
        col_sums = [sum(cm[i][j] for i in range(L)) for j in range(L)]

        out: dict[str, dict[str, float]] = {}
        for i, name in enumerate(labels):
            tp = cm[i][i]
            fp = col_sums[i] - tp
            fn = row_sums[i] - tp

            prec = tp / (tp + fp) if (tp + fp) else 0.0
            rec = tp / (tp + fn) if (tp + fn) else 0.0
            f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0

            out[name] = {
                "precision": prec,
                "recall": rec,
                "f1": f1,
                "support": float(row_sums[i]),
            }
        return out

### Experiments

#### Load data

Download and load the data from the following links

https://princeton-nlp.github.io/cos484/assignments/a2/eng.train

https://princeton-nlp.github.io/cos484/assignments/a2/eng.val

Then load the data using what you implemented

In [ ]:
# loads data and trains BigramHMM
import os
import urllib.request

# NOTE: loads in the train data again, since earlier cell was for testing only
TRAIN_URL = "https://princeton-nlp.github.io/cos484/assignments/a2/eng.train"
VALID_URL = "https://princeton-nlp.github.io/cos484/assignments/a2/eng.val"

TRAIN_PATH = "eng.train"
VALID_PATH = "eng.val"


def download(url: str, out_path: str) -> None:
    if os.path.exists(out_path):
        return
    urllib.request.urlretrieve(url, out_path)


# 1) download files
download(TRAIN_URL, TRAIN_PATH)
download(VALID_URL, VALID_PATH)

# 2) load + build vocab/tag maps from train
loader = NERDataLoader(normalize_digits=True)
loader.load(TRAIN_PATH, is_train=True)
loader.load(VALID_PATH, is_train=False)
loader.build_id_maps()

# 3) train HMM
trainer = HMMTrainer(loader, alpha_trans=1e-3, alpha_emit=1e-3)
model = trainer.train()

#### Experiment with an HMM with greedy decoding

In [ ]:
# instead of pure dict, use helper to cleanly print confusion matrix
# NOTE: called later after evaluating

def print_confusion_matrix(cm: list[list[int]], labels: list[str]) -> None:
    # simple pretty-print: fixed-width columns
    width = max(8, max(len(x) for x in labels) + 2)

    def cell(x):
        return str(x).rjust(width)

    # header
    print(" " * width + "".join(cell(l) for l in labels))
    for i, row in enumerate(cm):
        print(labels[i].rjust(width) + "".join(cell(v) for v in row))

# helper to print metrics
def print_per_class(res: dict) -> None:
    per = res["per_label"]
    labels = res["labels"]
    for lab in labels:
        m = per[lab]
        print(f"{lab:>5} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} supp={int(m['support'])}")

In [ ]:
# eval using greedy decoding
train_greedy = trainer.evaluate("train", decode="greedy")
valid_greedy = trainer.evaluate("valid", decode="greedy")

print("Greedy train acc:", train_greedy["accuracy"])
print("Greedy valid acc:", valid_greedy["accuracy"])

valid_cm = valid_greedy["confusion_matrix"]
train_cm = train_greedy["confusion_matrix"]
valid_labels = valid_greedy["labels"]
train_labels = train_greedy["labels"]

# print F1 and other eval metrics (per class, not aggregated)
print("Greedy train per-class:")
print_per_class(train_greedy)
print("Greedy valid per-class:")
print_per_class(valid_greedy)

# print the CM using helper
print("Training data confusion matrix for greedy:")
print_confusion_matrix(train_cm, train_labels)

print("Validation data confusion matrix for greedy:")
print_confusion_matrix(valid_cm, valid_labels)

Greedy train acc: 0.9839161972488103
Greedy valid acc: 0.9393717149492727
Greedy train per-class:
  ORG P=0.921 R=0.833 F1=0.874 supp=10025
    O P=0.995 R=0.997 F1=0.996 supp=169578
 MISC P=0.900 R=0.882 F1=0.891 supp=4593
  PER P=0.982 R=0.988 F1=0.985 supp=11128
  LOC P=0.885 R=0.941 F1=0.912 supp=8297
Greedy valid per-class:
  ORG P=0.649 R=0.653 F1=0.651 supp=2250
    O P=0.963 R=0.987 F1=0.975 supp=41164
 MISC P=0.773 R=0.691 F1=0.730 supp=1007
  PER P=0.950 R=0.640 F1=0.765 supp=2690
  LOC P=0.823 R=0.808 F1=0.815 supp=1975
Training data confusion matrix for greedy:
             ORG       O    MISC     PER     LOC
     ORG    8347     567     279     103     729
       O     244  169141     114      28      51
    MISC     111     192    4053      32     205
     PER      50      41       3   11000      34
     LOC     315      81      54      42    7805
Validation data confusion matrix for greedy:
             ORG       O    MISC     PER     LOC
     ORG    1469     456      63

**(a) Which pair of tags does the model have most difficulty separating according to the confusion matrix of the validation set?**

According to the CM of the validation set, O and ORG are often confused, since O (truth) + ORG (pred) pair has 357 wrong predictions, and ORG (truth) + O (pred) also has 456 mistakes. This tells us that between O and ORG, predictions are relatively more often mistaken compared to other tag pairs. Additionally, ORG, LOC pair are difficult to separate as well, since both LOC -> ORG (176) and ORG -> LOC (215) are high. Additionally, PER -> O pair (unidirectional) is also often confused with 745 mistakes, but not the converse.

Overall, there are a lot of dominating mistakes where tokens that are valid entities in reality are predicted as O. The accuracy is largely dominated by O predictions, which makes sense since the most occurence is O tag, as seen by 40629 O, O pairs existing.

#### Experiment with an HMM with viterbi decoding

In [ ]:
# eval using viterbi decoding
train_viterbi = trainer.evaluate("train", decode="viterbi")
valid_viterbi = trainer.evaluate("valid", decode="viterbi")

print("Viterbi train acc:", train_viterbi["accuracy"])
print("Viterbi valid acc:", valid_viterbi["accuracy"])

valid_cm = valid_viterbi["confusion_matrix"]
train_cm = train_viterbi["confusion_matrix"]
valid_labels = valid_viterbi["labels"]
train_labels = train_viterbi["labels"]

# print F1 and other metrics
print("Viterbi train per-class:")
print_per_class(train_viterbi)
print("Viterbi valid per-class:")
print_per_class(valid_viterbi)

# print the CM using helper
print("Training data confusion matrix for viterbi:")
print_confusion_matrix(train_cm, train_labels)

print("Validation data confusion matrix for viterbi:")
print_confusion_matrix(valid_cm, valid_labels)

Viterbi train acc: 0.9897652992569529
Viterbi valid acc: 0.9381493704926048
Viterbi train per-class:
  ORG P=0.922 R=0.929 F1=0.926 supp=10025
    O P=0.997 R=0.996 F1=0.997 supp=169578
 MISC P=0.936 R=0.952 F1=0.944 supp=4593
  PER P=0.992 R=0.994 F1=0.993 supp=11128
  LOC P=0.946 R=0.951 F1=0.949 supp=8297
Viterbi valid per-class:
  ORG P=0.755 R=0.729 F1=0.742 supp=2250
    O P=0.984 R=0.976 F1=0.980 supp=41164
 MISC P=0.404 R=0.785 F1=0.533 supp=1007
  PER P=0.939 R=0.660 F1=0.775 supp=2690
  LOC P=0.748 R=0.842 F1=0.792 supp=1975
Training data confusion matrix for viterbi:
             ORG       O    MISC     PER     LOC
     ORG    9318     235      92      34     346
       O     447  168888     167      30      46
    MISC      30     133    4373      12      45
     PER      28      20       1   11064      15
     LOC     284      66      38      15    7894
Validation data confusion matrix for viterbi:
             ORG       O    MISC     PER     LOC
     ORG    1641     243  

**(b) What major differences do you observe compared to the matrix in (a)**

Relative to the CM of the greedy decoding, there are fewer aggresion shown in predicting, as expected. Notably, the number of eager predictions for O is lessened, as seen that the wrong predictions for O are only (243, 114, 179, 125). This includes the O, ORG pair mistakes.

Additionally, the ORG, LOC pair is still confused often (249, 125) relatively, but compared to the earlier CM, this is much lower. However, PER -> MISC prediction is often difficult to separate, which wasn't observed before.

# LLM Prompts

If you used an AI tool to complete any part of this assignment, please paste all prompts you used to produce your final code/responses in the box below and answer the following reflection question.

Prompts Used:
*   How can I nicely print my confusion matrix from a python dict to something with row, col structure?
*   Given a url path for data, how can I download it? Should I use wget?
* Given my implementation of data loader, write simple test cases to verify it loaded correctly.
* Review my implementation of BigramHMM class and decoders for organization or minor optimizations. Is the core logic valid?
* What additional methods besides the F1 can be used to evaluate my results?



**Reflection: What parts of the AI generated output required modification or improvement? Describe the feedback you gave the tool to produce your final output or any changes you had to make on your own.**

Most of my changes were reviewing the tool's organizational improvements/optimizations to my code and making sure that the core logic was in-tact with the new changes suggested by AI. I also reviewed the matrix printing/formatting helper method generated to understand how it worked.